# Sensitivity of the chamber sizing

Which inputs actually move the numbers we are going to buy machines against,
and which ones only look like they do.

Three questions, kept separate because they have different answers:

| target | the question |
| --- | --- |
| `cooling_design` | how big is the machine |
| `heating_design` | how big is the *other* machine, which for the chamber is a different scenario |
| `cooling_operating`, `heating_operating` | what holding an experiment alone asks for |
| `cooling_ramp`, `heating_ramp` | what ramping alone asks for |
| `fastest_ramp_minutes` | how fast can the room actually go |

The design capacity is the larger of the operating and ramp requirements, never
their sum, so the two modes are screened separately as well: a parameter can
dominate the design number only by dominating whichever mode is setting it.

Two methods, and the disagreement between them is a result in its own right.
**Sobol** samples the whole space and splits each factor's share of the output
variance into what it does alone (`S1`) and what it does in company
(`ST − S1`). **One-at-a-time** moves each factor from the baseline with
everything else held still, which is what most people picture when they say
"sensitivity" — and which is structurally blind to anything that only matters
in combination.

Everything comes from `study/rooms.yaml`. Nothing below hardcodes a room.

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from zcbsl_resize import compute, fastest_ramp_minutes, sensitivity, study  # noqa: E402

FIGURES = REPO / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

config = study.load(REPO / "study" / "rooms.yaml")
ROOMS = list(config.rooms.values())
TARGETS = list(config.sobol.targets)
print(config.title)
for room in ROOMS:
    print(f"  {room.label}: {len(room.varying)} factors, "
          f"{room.interior_area:,.0f} m2 interior surface")

## Methods

Each room is a six-surface shoebox evaluated at two design conditions. A ramp
between setpoints is a mass-charging problem and holding a setpoint is a
steady-state one. The room is always doing one or the other, so the design
capacity is the larger of the two, each with its margin; the ramp carries the
hold at its own far end, evaluated with nobody inside and the Artificial Sun
off. Heat
reaches the thermal mass only across an air-to-surface film, which caps the
charging rate independently of how large the coil is, so a ramp can be
unreachable at any capacity. Sensitivity is measured by variance decomposition:
a Saltelli sample over every parameter the study config declares as varying,
evaluated analytically (the model is closed-form and vectorised, so ~10^5 cases
run in seconds), giving each parameter a first-order share of output variance
and a total-order share that includes every interaction it takes part in. The
difference between the two is the part a one-at-a-time sweep cannot recover.
Parameters the rooms *know* — geometry, the aluminium lining's capacitance, the
chamber's installed envelope — are held at their measured values and excluded
from the sample; the Artificial Sun is treated as a scenario rather than a
continuous input.

### The model

Total heat capacity, with only the depth the ramp reaches participating in any
added lining:

$$
C(t) \;=\; \underbrace{\rho_a V c_{p}}_{\text{air}}
\;+\; \underbrace{A_{\mathrm{int}}\, c_{\mathrm{shell}}}_{\text{lining as built}}
\;+\; \underbrace{A_{\mathrm{add}}\, \rho c\, d_{\mathrm{eff}}(t)}_{\text{added mass}},
\qquad
d_{\mathrm{eff}}(t) \;=\; \min\!\left( \frac{4}{3\sqrt{\pi}}\sqrt{\alpha t},\; L \right),
\quad \alpha = \frac{k}{\rho c}.
$$

Charging power, and the two modes. The operating hold carries the experiment's
internal gains $G$; the hold at the far end of a ramp carries only $G_r$, the
equipment left running (for the chamber, nothing: the Sun is off):

$$
P_{\mathrm{mass}}(t) = \frac{C(t)\,\Delta T_{\mathrm{ramp}}}{t},
\qquad
Q^{\mathrm{op}} = Q^{\mathrm{hold}}(G)\,(1+m),
\qquad
Q^{\mathrm{ramp}}(t) = \bigl(Q^{\mathrm{hold}}(G_r) + P_{\mathrm{mass}}(t)\bigr)(1+m),
$$

and for each duty the design capacity is

$$
Q_{\mathrm{design}} = \max\bigl(Q^{\mathrm{op}},\; Q^{\mathrm{ramp}}(t)\bigr).
$$

Hung radiant panels carry up to their flux limit of either; the air coil carries
the remainder and sets the supply airflow. Neither changes the totals above.

The film constraint. Heat crosses into the mass over the interior surface only,
so the air must run ahead of the surfaces by

$$
\Delta T_{\mathrm{req}}(t) = \frac{P_{\mathrm{mass}}(t)}{h\,A_{\mathrm{int}}}
\;\le\; \Delta T_{\mathrm{allow}} .
$$

With no added mass, $C$ is constant and $A_{\mathrm{int}}$ cancels top and
bottom — which is why ramp feasibility does not depend on room size. With added
mass it still cancels, **provided the lining is counted as a fraction of
interior surface**, $A_{\mathrm{add}} = \phi A_{\mathrm{int}}$.

The speed limit is where that binds. Because $C$ itself grows with $t$, this is
a fixed point rather than a threshold:

$$
t^{*}\,h\,A_{\mathrm{int}}\,\Delta T_{\mathrm{allow}} \;=\; \Delta T_{\mathrm{ramp}}\,C(t^{*}).
$$

Before the layer is fully penetrated this is a quadratic in $u=\sqrt{t^{*}}$,
with $P_{\max}=h A_{\mathrm{int}} \Delta T_{\mathrm{allow}}$,
$C_0$ the ramp-independent capacity and
$B = A_{\mathrm{add}}\,\rho c\,\tfrac{4}{3\sqrt{\pi}}\sqrt{\alpha}$:

$$
P_{\max} u^{2} - \Delta T_{\mathrm{ramp}} B\, u - \Delta T_{\mathrm{ramp}} C_{0} = 0
\quad\Longrightarrow\quad
t^{*} = \left( \frac{\Delta T_{\mathrm{ramp}} B + \sqrt{\Delta T_{\mathrm{ramp}}^{2} B^{2} + 4 P_{\max} \Delta T_{\mathrm{ramp}} C_{0}}}{2 P_{\max}} \right)^{\!2}.
$$

Once $\tfrac{4}{3\sqrt{\pi}}\sqrt{\alpha t^{*}} \ge L$ the capacity stops
moving and $t^{*} = \Delta T_{\mathrm{ramp}} C_{\mathrm{sat}} / P_{\max}$.

### The indices

For output $Y=f(X_1,\dots,X_k)$ with independent inputs:

$$
S_i = \frac{\operatorname{Var}_{X_i}\!\bigl(\mathbb{E}[Y \mid X_i]\bigr)}{\operatorname{Var}(Y)},
\qquad
S_{Ti} = \frac{\mathbb{E}_{X_{\sim i}}\!\bigl[\operatorname{Var}(Y \mid X_{\sim i})\bigr]}{\operatorname{Var}(Y)},
\qquad
S_{Ti} - S_i \;=\; \text{interaction}.
$$

Estimated from two independent samples $A,B$ of size $N$ and the matrices
$A_B^{(i)}$ (that is $A$ with column $i$ taken from $B$), by Saltelli *et al.*
(2010) and Jansen (1999) respectively:

$$
\hat{S}_i = \frac{\frac{1}{N}\sum_{j=1}^{N} f(B)_j \left[ f\!\left(A_B^{(i)}\right)_j - f(A)_j \right]}{\operatorname{Var}(Y)},
\qquad
\hat{S}_{Ti} = \frac{\frac{1}{2N}\sum_{j=1}^{N} \left[ f(A)_j - f\!\left(A_B^{(i)}\right)_j \right]^{2}}{\operatorname{Var}(Y)} .
$$

Cost is $N(k+2)$ evaluations. $\hat{S}_i$ can come out slightly negative for a
parameter with no effect; that is estimator noise, not a negative share.

## House style

Muted and earthy, legible in print, and one palette for the whole notebook so
the figures read as a set.

In [ ]:
EARTH = ["#8a6f4e", "#6f7f5c", "#a3705c", "#5f7480", "#9c8a5a", "#7a6070"]
INK = "#2e2a26"
PAPER = "#f9f9f9"
MUTED = "#9a9188"

SEQUENTIAL = LinearSegmentedColormap.from_list(
    "earth_sequential",
    ["#f4ece0", "#e0cba8", "#c69b6d", "#a3705c", "#6d4433", "#3d2620"],
)

mpl.rcParams.update({
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER,
    "savefig.facecolor": PAPER,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "axes.titlesize": 10,
    "axes.titleweight": "semibold",
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.frameon": False,
    "font.size": 9,
    "grid.color": "#e3ddd3",
    "grid.linewidth": 0.6,
    "figure.dpi": 110,
    "pdf.fonttype": 42,
})

PRETTY = {
    "cooling_design": "Cooling design capacity",
    "heating_design": "Heating design capacity",
    "cooling_operating": "Cooling, operating",
    "heating_operating": "Heating, operating",
    "cooling_ramp": "Cooling, ramp",
    "heating_ramp": "Heating, ramp",
    "fastest_ramp_minutes": "Fastest reachable ramp",
}

def label(key):
    """Parameter keys, made readable without losing which surface they are on."""
    return key.replace("_", " ").replace("u opaque", "U opaque").replace("wwr", "WWR")

def save(fig, index, slug):
    path = FIGURES / f"fig_{index}_{slug}.pdf"
    fig.savefig(path, bbox_inches="tight")
    print("wrote", path.relative_to(REPO))
    return path

## Scenarios

The Artificial Sun is 10.5 kW of gain that is either on or off during an
experiment. It is a separate world to screen, not a slider to sample across,
so it is pinned and dropped from the chamber's factor list.

It only reaches the **operating** mode. The Sun is off during every ramp, so
the ramp requirement is the same whichever way an experiment runs it. For the
operating hold it matters a great deal: with the Sun on the chamber barely
needs heating, and with it off the heating hold is at its largest. So each
duty's operating requirement is screened in its own worst case (heating Sun
off, cooling Sun on), and the ramp targets and the speed limit are screened
once, with the Sun pinned only so it drops out of the factor list.

In [ ]:
from zcbsl_resize import rooms as room_defs  # noqa: E402

# 10 500 W of light over the chamber's 72.45 m2 floor, read off rooms.py so
# this cannot drift from the geometry again.
SUN_ON = room_defs.ARTIFICIAL_SUN_W / (10.5 * 6.9)
SUN_OFF = 0.0

HEATING_WORST = {"heating_design", "heating_operating"}

def scenarios_for(room, target):
    """Which world each room/target pair is screened in.

    The module room has no Sun. For the chamber, the operating mode and the
    design value are screened in each duty's worst case: Sun off for heating,
    Sun on for cooling. The ramp targets and the speed limit do not see the Sun
    at all -- it is off during every ramp, and internal gains never enter the
    film limit -- so one scenario is enough.
    """
    if room.key != "climate_chamber":
        return {"": None}
    if target in HEATING_WORST:
        return {"sun off": {"equipment_w_per_m2": SUN_OFF}}
    return {"sun on": {"equipment_w_per_m2": SUN_ON}}

rows = []
for room in ROOMS:
    for eq, name in ((SUN_ON, "sun on"), (SUN_OFF, "sun off")):
        if room.key != "climate_chamber" and eq == SUN_ON:
            continue
        params = room.nominal()
        if room.key == "climate_chamber":
            params = params.replace(equipment_w_per_m2=eq)
        r = compute(params)
        rows.append({
            "room": room.label,
            "scenario": name if room.key == "climate_chamber" else "-",
            "heating operating kW": float(r["heating_operating"]) / 1000,
            "heating ramp kW": float(r["heating_ramp"]) / 1000,
            "heating design kW": float(r["heating_design"]) / 1000,
            "cooling operating kW": float(r["cooling_operating"]) / 1000,
            "cooling ramp kW": float(r["cooling_ramp"]) / 1000,
            "cooling design kW": float(r["cooling_design"]) / 1000,
            "fastest ramp min": float(fastest_ramp_minutes(params)),
        })
scenario_table = pd.DataFrame(rows).round(2)
scenario_table

**Read the chamber's two rows against each other.** Only the operating columns
move with the Sun: heating operating falls by the Sun's gain when it is on, and
cooling operating rises by it. The ramp columns are identical, because the Sun
is off during every ramp. Whichever mode is larger sets the design column, so
the Sun can only change a design value when the operating mode is the one
setting it.

## The Sobol screen

`n = 4096` per room, `n x (k + 2)` evaluations — about 86 000 cases for the
module room, 61 000 for the chamber, a couple of seconds each.

In [ ]:
results = {}
for room in ROOMS:
    for target in TARGETS:
        for name, fixed in scenarios_for(room, target).items():
            key = (room.key, target, name)
            results[key] = sensitivity.screen(
                room, targets=[target], n=config.sobol.n,
                seed=config.sobol.seed, fixed=fixed,
            )[target]

for (room_key, target, name), r in results.items():
    tag = f"{room_key}/{target}" + (f" [{name}]" if name else "")
    print(f"{tag:52s} k={len(r.factors):2d}  {r.evaluations:,} evals")

In [ ]:
def ranking(room_key, target, scenario=None, top=8):
    matches = [k for k in results if k[0] == room_key and k[1] == target
               and (scenario is None or k[2] == scenario)]
    return results[matches[0]].table(top=top)

ranking("module_room", "cooling_design")

In [ ]:
ranking("module_room", "fastest_ramp_minutes")

### What ranks where

The bar is the total-order share `ST`. The solid part is what the factor does
on its own; the pale part on top is interaction. A bar that is mostly pale is a
factor you cannot understand by moving it alone.

In [ ]:
def ranking_panel(ax, result, top=7, title=""):
    frame = result.table(top=top).iloc[::-1]
    y = np.arange(len(frame))
    ax.barh(y, frame["S1"], color=EARTH[0], height=0.62, label="alone (S1)")
    ax.barh(y, frame["interaction"], left=frame["S1"], height=0.62,
            color=EARTH[0], alpha=0.34, label="in company (ST - S1)")
    ax.set_yticks(y)
    ax.set_yticklabels([label(f) for f in frame["factor"]])
    ax.set_xlim(0, max(0.35, float(frame["ST"].max()) * 1.18))
    ax.set_xlabel("share of variance")
    ax.set_title(title, loc="left")
    ax.grid(axis="x", zorder=0)
    ax.set_axisbelow(True)

# The headline numbers only; the operating/ramp split is read off the
# influence matrix (fig 6), where fourteen columns fit and fourteen panels would not.
HEADLINE = [t for t in ("cooling_design", "heating_design", "fastest_ramp_minutes") if t in TARGETS]
panels = []
for room in ROOMS:
    for target in HEADLINE:
        for name in scenarios_for(room, target):
            panels.append((room, target, name))

fig, axes = plt.subplots(len(ROOMS), len(HEADLINE), figsize=(4.5 * len(HEADLINE), 3.6 * len(ROOMS)))
for ax, (room, target, name) in zip(axes.flat, panels):
    r = results[(room.key, target, name)]
    suffix = f"  ({name})" if name else ""
    ranking_panel(ax, r, title=f"{room.label}\n{PRETTY[target]}{suffix}")
axes.flat[0].legend(loc="lower right")
fig.suptitle("Total-order Sobol indices, split into own effect and interaction",
             x=0.005, ha="left", fontsize=11, fontweight="semibold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, 1, "sobol_factor_ranking")

## Where one-at-a-time would have sent you wrong

Both rooms start with no added lining, and a one-at-a-time sweep moves each
factor out from there with everything else held still. Two consequences:

- The lining's **thickness** moves the answer by exactly zero, because there is
  no lining for a thickness to describe. Sobol samples area and thickness
  together, finds that they multiply, and gives it a small but real
  total-order share that is almost entirely interaction.
- The two methods **swap first place**. One at a time, the biggest single lever
  on the ramp is how much lining you install. Sampling the whole space, it is
  the air-to-surface ΔT you are willing to run — which is also the number
  nobody has measured.

Neither ranking is wrong. They answer different questions, and the design
question is the second one.

In [ ]:
room = config.rooms["module_room"]
target = "fastest_ramp_minutes"

oat_frame = sensitivity.oat(room, targets=[target], points=41).set_index("factor")
sobol_frame = results[(room.key, target, "")].table().set_index("factor")

comparison = pd.DataFrame({
    "OAT swing (min)": oat_frame[f"{target}_swing"],
    "Sobol ST": sobol_frame["ST"],
    "interaction": sobol_frame["interaction"],
}).sort_values("Sobol ST", ascending=False)
comparison.round(4).head(10)

In [ ]:
shown = comparison.head(8).iloc[::-1].copy()
shown["oat_share"] = shown["OAT swing (min)"] / shown["OAT swing (min)"].max()
shown["sobol_share"] = shown["Sobol ST"] / shown["Sobol ST"].max()

y = np.arange(len(shown))
height = 0.38

fig, ax = plt.subplots(figsize=(8.8, 5.4))
ax.barh(y + height / 2, shown["oat_share"], height, color=MUTED,
        label="one at a time: swing from the baseline")
ax.barh(y - height / 2, shown["sobol_share"], height, color=EARTH[0],
        label="Sobol: total-order share")

for yi, (_, row) in zip(y, shown.iterrows()):
    if row["OAT swing (min)"] <= 1e-9:
        ax.annotate("moves nothing on its own", (0.012, yi + height / 2), va="center",
                    fontsize=8, color=EARTH[2], fontstyle="italic")

ax.set_yticks(y)
ax.set_yticklabels([label(f) for f in shown.index])
ax.set_xlim(0, 1.06)
ax.set_xlabel("share of the largest, within each method")
ax.set_title(f"{room.label} · {PRETTY[target]}\n"
             "the two methods do not agree on what matters most", loc="left")
ax.legend(loc="lower right")
ax.grid(axis="x")
ax.set_axisbelow(True)

top_oat = shown["OAT swing (min)"].idxmax()
top_sobol = shown["Sobol ST"].idxmax()
if top_oat != top_sobol:
    ax.annotate(f"one at a time puts {label(top_oat)} first;\n"
                f"sampling the whole space puts {label(top_sobol)} first",
                xy=(0.33, 0.30), xycoords="axes fraction", fontsize=8, color=INK)
fig.tight_layout()
save(fig, 2, "oat_versus_sobol")

## The feasibility surface

Ramp time against lining, as a share of interior surface. Colour is the
air-to-surface ΔT the ramp demands; the line is where that crosses the
allowance and the ramp becomes reachable.

**The boundary is drawn from `film_ok`, not from
`min_feasible_ramp_minutes`.** That output is conditional on the ramp it was
handed: a longer ramp lets heat diffuse deeper, recruits more mass, and moves
the answer. `film_ok` compares the required ΔT at *that* ramp against the
allowance, so it flips exactly at the self-consistent fixed point that
`fastest_ramp_minutes` solves for.

Note the two panels are nearly the same picture. That is the point of counting
lining as a fraction: the area cancels out of capacity-over-film-conductance,
so ramp feasibility is a question about the construction, asked once — not a
question about the room.

In [ ]:
ramp_axis = np.linspace(10, 240, 200)
coverage_axis = np.linspace(0.0, 0.50, 160)
LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 60]

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.9), sharey=True, sharex=True)
filled = None
for ax, room in zip(axes, ROOMS):
    fixed = {"equipment_w_per_m2": SUN_ON} if room.key == "climate_chamber" else None
    grid = sensitivity.feasibility_grid(
        room, ramp_minutes=ramp_axis, coverage=coverage_axis, fixed=fixed
    )
    required = grid["required_air_surface_dt"]
    allowance = float(room.nominal().max_air_surface_dt)
    y = coverage_axis * 100

    filled = ax.contourf(ramp_axis, y, required, levels=LEVELS,
                         cmap=SEQUENTIAL, extend="max")
    lines = ax.contour(ramp_axis, y, required, levels=LEVELS[1:-1],
                       colors=[INK], linewidths=0.5, alpha=0.35)
    ax.clabel(lines, fmt="%d K", fontsize=7, inline=True)

    # Everything left of the boundary is unreachable: wash it back so the eye
    # goes to the part of the space you can actually operate in.
    ax.contourf(ramp_axis, y, (required > allowance).astype(float),
                levels=[0.5, 1.5], colors=[PAPER], alpha=0.55)
    ax.contour(ramp_axis, y, required, levels=[allowance],
               colors=[INK], linewidths=2.0)

    # The bare room, which is the number in the sizing table.
    bare = float(fastest_ramp_minutes(room.nominal()))
    ax.plot([bare], [0], marker="o", ms=6, color=INK, zorder=5, clip_on=False)
    ax.annotate(f"bare shell: {bare:.0f} min", (bare, 0), textcoords="offset points",
                xytext=(9, 7), fontsize=8, color=INK)

    ax.set_xlabel("ramp time (min)")
    ax.set_title(f"{room.label} · {room.interior_area:,.0f} m² interior", loc="left")

axes[0].set_ylabel("rammed-earth lining, % of interior surface")
axes[0].set_xlim(ramp_axis.min(), ramp_axis.max())
axes[0].set_ylim(0, 50)
axes[0].annotate("washed out: the film cannot deliver this ramp\nbold line: the allowance, "
                 f"{float(ROOMS[0].nominal().max_air_surface_dt):.0f} K",
                 xy=(0.44, 0.06), xycoords="axes fraction", fontsize=8, color=INK)

cbar = fig.colorbar(filled, ax=axes, fraction=0.03, pad=0.02, ticks=LEVELS)
cbar.set_label("air-to-surface ΔT the ramp demands (K)")
cbar.outline.set_visible(False)
fig.suptitle("Ramp feasibility is a question about the construction, not the room",
             x=0.005, ha="left", fontsize=11, fontweight="semibold")
save(fig, 3, "ramp_feasibility_surface")

In [ ]:
# The same thing as a table, for the rooms as they would actually be built.
rows = []
for room in ROOMS:
    for share in (0.0, 0.10, 0.25, 0.50):
        params = room.nominal().replace(added_mass_coverage=share * 100.0)
        rows.append({
            "room": room.label,
            "lining %": int(share * 100),
            "lining m2": round(share * room.interior_area),
            "fastest ramp (min)": round(float(fastest_ramp_minutes(params)), 1),
        })
pd.DataFrame(rows).pivot(index="lining %", columns="room", values="fastest ramp (min)")

## The heat pump

Lift and COP fall out of the setpoints and the tank temperatures alone, so both
rooms sit on the same curve — only the electrical input differs, in proportion
to the duty. The question the left panel answers is whether free cooling is
ever available: it would need the required supply-air temperature to sit
*above* the cold tank plus the exchanger approach, and across the whole useful
setpoint range it does not. The cooling duty needs a real machine, not a valve.

In [ ]:
setpoints = np.linspace(8.0, 24.0, 160)
reference = ROOMS[0].nominal()
tank_floor = float(reference.tank_temp_cold) + float(reference.exchanger_approach)

fig, axes = plt.subplots(1, 3, figsize=(13.4, 4.3))
common = compute(reference.replace(setpoint_min=setpoints))

axes[0].plot(setpoints, np.asarray(common["supply_temp_cooling"]), color=EARTH[0], lw=1.9,
             label="required supply air")
axes[0].axhline(tank_floor, color=EARTH[2], lw=1.4, ls=":",
                label=f"cold tank + approach = {tank_floor:.0f} °C")
axes[0].fill_between(setpoints, np.asarray(common["supply_temp_cooling"]), tank_floor,
                     where=np.asarray(common["supply_temp_cooling"]) < tank_floor,
                     color=EARTH[2], alpha=0.12)
axes[0].set_ylabel("°C")
axes[0].set_title("Free cooling: how far short the tank falls", loc="left")
axes[0].legend(loc="lower right")
gap = tank_floor - float(np.max(np.asarray(common["supply_temp_cooling"])))
axes[0].annotate(f"closest approach {gap:.1f} K short, at the warm end",
                 xy=(0.04, 0.06), xycoords="axes fraction", fontsize=8, color=INK)

axes[1].plot(setpoints, common["cop_cooling"], color=EARTH[0], lw=1.9)
axes[1].set_ylabel("COP")
axes[1].set_title("Cooling COP — the same for both rooms", loc="left")
axes[1].annotate("Carnot × efficiency over a small lift.\nAn upper bound, not a machine curve.",
                 xy=(0.04, 0.82), xycoords="axes fraction", fontsize=8, color=MUTED)

for room, colour in zip(ROOMS, (EARTH[0], EARTH[3])):
    params = room.nominal()
    if room.key == "climate_chamber":
        params = params.replace(equipment_w_per_m2=SUN_ON)
    r = compute(params.replace(setpoint_min=setpoints))
    axes[2].plot(setpoints, np.asarray(r["electric_cooling"]) / 1000, color=colour, lw=1.9,
                 label=room.label)
axes[2].set_ylabel("kW")
axes[2].set_title("Electrical input, cooling", loc="left")
axes[2].legend(loc="upper right")

for ax in axes:
    ax.set_xlabel("cold setpoint (°C)")
    ax.set_xlim(setpoints.min(), setpoints.max())
    ax.grid(True)
    ax.set_axisbelow(True)

assert not np.any(np.asarray(common["free_cooling"])), "free cooling turned out reachable"
fig.suptitle("The cooling duty needs a chiller, not a valve",
             x=0.005, ha="left", fontsize=11, fontweight="semibold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
save(fig, 4, "heat_pump_duty")

## Design capacity by scenario

Both modes, per duty. The taller bar of each pair is the design value; the two
are never added.

In [ ]:
fig, ax = plt.subplots(figsize=(9.6, 4.6))

rows = scenario_table.copy()
rows["name"] = rows["room"] + rows["scenario"].map(lambda s: "" if s == "-" else f"\n({s})")
x = np.arange(len(rows))
width = 0.2
bars = [
    ("heating operating kW", EARTH[0], 0.45, "heating, operating"),
    ("heating ramp kW", EARTH[0], 1.0, "heating, ramp"),
    ("cooling operating kW", EARTH[3], 0.45, "cooling, operating"),
    ("cooling ramp kW", EARTH[3], 1.0, "cooling, ramp"),
]
for n, (col, colour, alpha, name) in enumerate(bars):
    offset = (n - 1.5) * width
    ax.bar(x + offset, rows[col], width, color=colour, alpha=alpha, label=name)
    duty = col.split()[0]
    for xi, (v, design) in enumerate(zip(rows[col], rows[f"{duty} design kW"])):
        governs = abs(v - design) < 1e-6
        ax.annotate(f"{v:.1f}", (xi + offset, v), ha="center", va="bottom", fontsize=7,
                    fontweight="bold" if governs else "normal")

ax.set_xticks(x)
ax.set_xticklabels(rows["name"])
ax.set_ylabel("kW, room total incl. margin")
ax.set_title("Operating vs ramp per duty; the taller of each pair is what gets bought", loc="left")
ax.legend(ncols=2)
ax.grid(axis="y")
ax.set_axisbelow(True)
fig.tight_layout()
save(fig, 5, "design_capacity_by_scenario")

## What actually drives each room

Everything below is generated from the run above, so it cannot go stale.

First, what is *not* in play. Each room's geometry, its aluminium lining, and
for the chamber its installed envelope, are measured values held fixed — they
are not uncertain, so they cannot be influential. What remains varying is how
the rooms are *operated* and what gets *added* to them.

In [ ]:
fixed_rows = []
for room in ROOMS:
    varying = set(room.varying)
    measured = sorted(__import__("zcbsl_resize").rooms.MEASURED[room.base])
    fixed_here = sorted(k for k, spec in room.specs.items() if not spec.varies)
    fixed_rows.append({
        "room": room.label,
        "varying": len(varying),
        "held at a measured value": len(measured),
        "fixed by choice in the config": len(fixed_here),
    })
pd.DataFrame(fixed_rows).set_index("room")

In [ ]:
BANDS = [(0.40, "dominant"), (0.15, "material"), (0.05, "minor"), (0.0, "negligible")]

def band(st):
    for threshold, name in BANDS:
        if st >= threshold:
            return name
    return "negligible"

matrix = {}
for (room_key, target, scenario), r in results.items():
    room_label = config.rooms[room_key].label
    col = f"{room_label}\n{PRETTY[target]}" + (f" ({scenario})" if scenario else "")
    matrix[col] = pd.Series(r.total_order, index=[label(f) for f in r.factors])

influence = pd.DataFrame(matrix).fillna(0.0)
influence = influence.loc[influence.max(axis=1).sort_values(ascending=False).index]
influence.round(3)

Read down a column for one room-and-duty; read across a row to see whether
a parameter matters everywhere or only in one place. Values are total-order
shares of output variance — a parameter at 0.50 accounts for half the spread in
that number, counting its interactions.

In [ ]:
import textwrap

fig, ax = plt.subplots(figsize=(max(10.4, 1.0 * influence.shape[1] + 3.5), 0.46 * len(influence) + 2.6))
mesh = ax.imshow(influence.to_numpy(), cmap=SEQUENTIAL, vmin=0, vmax=0.7, aspect="auto")

ax.set_xticks(range(influence.shape[1]))
ax.set_xticklabels(["\n".join(textwrap.wrap(col.replace("\n", " "), 15)) for col in influence.columns],
                   fontsize=6.5, ha="center")
ax.set_yticks(range(influence.shape[0]))
ax.set_yticklabels(influence.index, fontsize=8)
ax.set_xticks(np.arange(-0.5, influence.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, influence.shape[0], 1), minor=True)
ax.grid(which="minor", color=PAPER, linewidth=1.6)
ax.tick_params(which="minor", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

for i in range(influence.shape[0]):
    for j in range(influence.shape[1]):
        v = influence.iat[i, j]
        if v >= 0.02:
            ax.annotate(f"{v:.2f}", (j, i), ha="center", va="center", fontsize=7.5,
                        color=PAPER if v > 0.38 else INK)

cbar = fig.colorbar(mesh, ax=ax, fraction=0.022, pad=0.015)
cbar.set_label("total-order share of variance")
cbar.outline.set_visible(False)
ax.set_title("What moves each number, once the measured values are held fixed",
             loc="left", fontsize=11, pad=14)
fig.tight_layout()
save(fig, 6, "influence_matrix")

### In words

In [ ]:
def phrase(room_key, target, scenario):
    r = results[(room_key, target, scenario)]
    frame = r.table()
    lead = frame.iloc[0]
    strong = frame[frame["ST"] >= 0.15]
    weak = frame[frame["ST"] < 0.05]

    parts = [
        f"{band(lead['ST'])}: {label(lead['factor'])} (ST {lead['ST']:.2f}, "
        f"{lead['interaction'] / lead['ST']:.0%} of it interaction)"
    ]
    rest = strong.iloc[1:]
    if len(rest):
        parts.append("then " + ", ".join(
            f"{label(row.factor)} ({row.ST:.2f})" for row in rest.itertuples()
        ))
    parts.append(f"{len(weak)} of {len(frame)} parameters land below 0.05 and can be "
                 "set from judgement without moving the answer")
    return "; ".join(parts)

for room in ROOMS:
    print(room.label.upper())
    for target in TARGETS:
        for scenario in scenarios_for(room, target):
            key = (room.key, target, scenario)
            if key not in results or results[key].variance <= 0:
                continue
            tag = PRETTY[target] + (f" ({scenario})" if scenario else "")
            print(f"  {tag}")
            print(f"    {phrase(*key)}")
    print()

### The short version

- **Split by mode, the drivers separate cleanly.** The operating hold answers to
  what the room is exposed to and how it is run: for the module room the facade
  (glazed share, irradiance, SHGC) sets the cooling hold and the warm setpoint
  and air-change rate set the heating hold; for the chamber, whose envelope is
  fixed, the setpoints alone. The ramp answers to ramp time and added mass,
  and to almost nothing else.
- **At today's ranges the ramp sets every design number.** Ramp time leads
  every design column, and each design column is a near-copy of its ramp
  column. The module room's exchangeable facade, dominant in the operating
  mode, barely registers in the design value, because the operating mode is
  not the one setting it. That changes only where the ramp is long enough, or
  the operating load large enough, for the operating mode to take over. The
  browser tool reports that crossover ramp time for any scenario.
- **The ramp is set by the film allowance and the lining, together.** Neither
  is decided yet, and they interact strongly enough that neither can be chosen
  on its own.
- **The chamber's added mass has stopped mattering, by choice.** Its coverage
  range is now 0-10 % (from 0-100 %), so it carries about 3 % of the
  chamber's design variance against about 30 % in the module room. The two
  rooms would still behave alike at equal coverage; they just are not being
  asked the same question any more.
- **The long tail is genuinely a long tail.** Most parameters sit below 0.05 in
  every column. They still need sensible values, but they do not need to be
  argued about.

### Standing caveats

- `max_air_surface_dt` is the decisive unknown and it is not a measured number.
  It tops the ramp ranking, and mostly through interaction, so it cannot be
  pinned down by moving it on its own.
- Every exposed surface takes its peak irradiance simultaneously in the cooling
  case. For the chamber's three exposed walls that is pessimistic; its cooling
  figure is an upper bound.
- The lining double-counts the 7 kJ/m²K shell beneath it, about 13% on the
  lined area, in the conservative direction.
- COP is Carnot × efficiency over very small lifts, which flatters a real
  machine. Thermal sizing is unaffected.